# Proyecto: Control de Spotify por reconocimiento de gestos

Análisis exploratorio, limpieza, partición estratificada (70/15/15), comparación de clasificadores sobre 21 landmarks 3D de la mano y evaluación del modelo para la demo en vivo.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Sube desde notebooks/ hasta la carpeta que contiene data/ y scripts/.
RAIZ = Path.cwd()
for carpeta in (RAIZ, *RAIZ.parents):
    if (carpeta / "data" / "dataset.csv").exists():
        RAIZ = carpeta
        break
sys.path.append(str(RAIZ))

from scripts.utils import (
    CLASES_GESTOS,
    COLUMNAS_FEATURES,
    cargar_dataset,
    construir_matriz_x,
    dividir_estratificado,
    limpiar_dataset,
)

## 1. Carga y exploración del dataset

Cargamos el archivo `data/dataset.csv`, que contiene las 63 coordenadas 3D de los 21 puntos de la mano (`x0..x20`, `y0..y20`, `z0..z20`) junto con la etiqueta del gesto y metadatos de captura.

In [2]:
RUTA_DATASET = RAIZ / "data" / "dataset.csv"
df = cargar_dataset(RUTA_DATASET)

print(f"Dataset: {RUTA_DATASET}")
print(f"Filas: {len(df)} | Columnas: {df.shape[1]}")
print(f"Features: {len(COLUMNAS_FEATURES)}")
df.head(3)

Dataset: /home/kaleb/Escuela/Minería/Proyecto 2do Parc/data/dataset.csv
Filas: 2400 | Columnas: 67
Features: 63


,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,z15,z16,z17,z18,z19,z20,label,handedness,session_id,timestamp
0,0.0,-0.262239,-0.455679,-0.549445,-0.643488,-0.225498,-0.276581,-0.296054,-0.309285,-0.054652,...,-0.240040,-0.284329,-0.157218,-0.221438,-0.261125,-0.293012,palma,Left,6e1ead7e,1.794529e+09
1,0.0,-0.255550,-0.444138,-0.533219,-0.628852,-0.216190,-0.266478,-0.285362,-0.296839,-0.046607,...,-0.242642,-0.286766,-0.160484,-0.224461,-0.262976,-0.293828,palma,Left,6e1ead7e,1.794529e+09
2,0.0,-0.251190,-0.434229,-0.521921,-0.616520,-0.211867,-0.261488,-0.282763,-0.297150,-0.046338,...,-0.246788,-0.287947,-0.168063,-0.229815,-0.264809,-0.292896,palma,Left,6e1ead7e,1.794529e+09


## 2. Calidad y balance de clases

Se revisan nulos, valores no finitos, duplicados exactos y la distribución de etiquetas y manos.

In [3]:
import numpy as np

X = df[COLUMNAS_FEATURES].to_numpy(dtype=float)
print(f"Valores nulos: {int(df.isna().sum().sum())}")
print(f"Valores no finitos: {int((~np.isfinite(X)).sum())}")
print(f"Duplicados exactos de features: {int(df.duplicated(subset=COLUMNAS_FEATURES).sum())}")

conteo = df["label"].value_counts().reindex(CLASES_GESTOS)
print("\nMuestras por clase:")
print(conteo.to_string())
print(f"\nProporción mínima: {conteo.min() / len(df):.2%}")
print(f"Proporción máxima: {conteo.max() / len(df):.2%}")
print("\nMuestras por clase y mano:")
print(df.groupby(["label", "handedness"]).size().unstack(fill_value=0).to_string())

Valores nulos: 0
Valores no finitos: 0
Duplicados exactos de features: 0

Muestras por clase:
label
palma      400
senalar    400
paz        400
pulgar     400
puno       400
neutro     400

Proporción mínima: 16.67%
Proporción máxima: 16.67%

Muestras por clase y mano:
handedness  Left  Right
label                  
neutro       400      0
palma        400      0
paz            0    400
pulgar         0    400
puno           0    400
senalar      400      0


## 3. Limpieza de datos

Verificamos que no existan valores nulos, infinitos, vectores degenerados, coordenadas fuera de rango ni filas duplicadas, comprobando que los faltantes queden en 0%.

In [4]:
limpio = limpiar_dataset(df)

print(f"Filas originales: {len(df)}")
print(f"Filas descartadas: {len(df) - len(limpio)}")
print(f"Filas limpias: {len(limpio)}")
assert len(limpio) == len(df), "La limpieza no debe perder muestras del dataset actual"
assert limpio["label"].value_counts().to_dict() == {clase: 400 for clase in CLASES_GESTOS}
print("DATASET_LIMPIO_OK")

Filas originales: 2400
Filas descartadas: 0
Filas limpias: 2400
DATASET_LIMPIO_OK


## 4. Partición estratificada 70 / 15 / 15

La semilla es fija (`random_state=42`) y la estratificación conserva las 6 clases en cada subconjunto.

In [5]:
split = dividir_estratificado(limpio)

for nombre in ("train", "val", "test"):
    X_parte = split[f"X_{nombre}"]
    y_parte = split[f"y_{nombre}"]
    print(f"{nombre:5s}: X={X_parte.shape} | y={y_parte.shape} | {len(y_parte) / len(limpio):.1%}")

tabla = pd.DataFrame(
    {
        nombre: pd.Series(split[f"y_{nombre}"]).value_counts().reindex(CLASES_GESTOS)
        for nombre in ("train", "val", "test")
    }
)
tabla

train: X=(1680, 63) | y=(1680,) | 70.0%
val  : X=(360, 63) | y=(360,) | 15.0%
test : X=(360, 63) | y=(360,) | 15.0%


,train,val,test
palma,280,60,60
senalar,280,60,60
paz,280,60,60
pulgar,280,60,60
puno,280,60,60
neutro,280,60,60


In [6]:
total = sum(len(split[f"y_{nombre}"]) for nombre in ("train", "val", "test"))
assert total == len(limpio), "Las particiones deben cubrir todas las filas"

for nombre in ("train", "val", "test"):
    proporciones = pd.Series(split[f"y_{nombre}"]).value_counts(normalize=True)
    assert proporciones.min() >= 0.14
    assert proporciones.max() <= 0.19

X_train = construir_matriz_x(limpio.iloc[:5])
assert X_train.shape == (5, 63)
print("PARTICION_ESTRATIFICADA_OK")

PARTICION_ESTRATIFICADA_OK


## Observación metodológica

Cada clase fue capturada en una sola sesión continua de ~13 s a ~30 FPS. Al partir aleatoriamente, cuadros consecutivos casi idénticos pueden quedar en `train` y `test`, lo que **sobreestima** las métricas en comparación con una persona nueva frente a la cámara.

Además, cada clase se grabó con una sola mano (tres clases con izquierda, tres con derecha). El espejado de `construir_matriz_x()` canonicaliza la lateralidad, pero conviene registrar ambas manos por clase en una versión futura del dataset.

Estas limitaciones deben declararse en el reporte; no invalidan el pipeline, pero sí acotan la interpretación del F1 obtenido.

## 5. Entrenamiento y comparacion de clasificadores

Comparamos cinco modelos clasicos en el conjunto de validacion: SVM con kernel RBF, SVM lineal, KNN, Random Forest y MLP. El conjunto de prueba se reserva intacto para la evaluacion final.

In [7]:
import json
import time
from datetime import datetime

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler

from scripts.modelado import entrenar_knn_final, entrenar_y_comparar, seleccionar_mejor

resultados, modelos = entrenar_y_comparar(
    split["X_train"], split["y_train"], split["X_val"], split["y_val"]
)
tabla = pd.DataFrame(resultados).T.sort_values("f1_macro", ascending=False)
tabla[["accuracy", "f1_macro", "latencia_ms", "tiempo_entrenamiento_s"]]


,accuracy,f1_macro,latencia_ms,tiempo_entrenamiento_s
svm_rbf,1.0,1.0,0.365584,0.01354
svm_lineal,1.0,1.0,0.363727,0.007626
random_forest,1.0,1.0,22.499346,0.956667
mlp,1.0,1.0,0.727342,0.145954
knn,1.0,1.0,1.157058,0.238818


In [8]:
mejor = seleccionar_mejor(resultados)
metricas = {
    "fecha": datetime.now().isoformat(timespec="seconds"),
    "semilla": 42,
    "dataset": {
        "total": len(limpio),
        "train": len(split["y_train"]),
        "val": len(split["y_val"]),
        "test": len(split["y_test"]),
        "clases": CLASES_GESTOS,
    },
    "modelos": resultados,
    "mejor_modelo": mejor,
    "modelo_seleccionado": "knn",
    "motivo_seleccion": "Seleccionamos KNN (k=3) por su simplicidad y baja latencia en CPU.",
}

RUTA_METRICAS = RAIZ / "models" / "metricas.json"
RUTA_METRICAS.parent.mkdir(parents=True, exist_ok=True)
RUTA_METRICAS.write_text(json.dumps(metricas, indent=2, ensure_ascii=False), encoding="utf-8")

print("Mejor modelo por F1 en validación:")
print(json.dumps(mejor, indent=2, ensure_ascii=False))
print(f"\nMétricas guardadas en {RUTA_METRICAS}")

Mejor modelo por F1 en validación:
{
  "nombre": "svm_lineal",
  "accuracy": 1.0,
  "f1_macro": 1.0,
  "latencia_ms": 0.36372704002133105,
  "tiempo_entrenamiento_s": 0.007625963000464253
}

Métricas guardadas en /home/kaleb/Escuela/Minería/Proyecto 2do Parc/models/metricas.json


In [9]:
escalador_pca = StandardScaler()
X_train_escalado = escalador_pca.fit_transform(split["X_train"])
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train_escalado)

fig, ax = plt.subplots(figsize=(8, 6))
mapa_colores = plt.get_cmap("tab10")
for indice, clase in enumerate(CLASES_GESTOS):
    mascara = split["y_train"] == clase
    ax.scatter(
        X_pca[mascara, 0], X_pca[mascara, 1],
        s=14, alpha=0.75, label=clase, color=mapa_colores(indice),
    )
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} de varianza)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} de varianza)")
ax.set_title("Proyección PCA 2D de los gestos (train)")
ax.legend(title="Gesto", ncol=2)
fig.tight_layout()

RUTA_PCA = RAIZ / "reporte" / "figuras" / "pca_2d.png"
RUTA_PCA.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(RUTA_PCA, dpi=150)
plt.close(fig)
print(f"Figura PCA guardada en {RUTA_PCA}")


Figura PCA guardada en /home/kaleb/Escuela/Minería/Proyecto 2do Parc/reporte/figuras/pca_2d.png


## 6. Evaluacion en el conjunto de prueba y Matriz de Confusion

Entrenamos **KNN (`k=3`)** con `train + val` y lo evaluamos una sola vez sobre el conjunto de prueba (`test`, 360 muestras) para obtener Accuracy, Precision, Recall, F1-Score y la matriz de confusion.

In [10]:
from scripts.utils import COLUMNAS_FEATURES

k_knn = int(resultados["knn"]["n_neighbors"])
modelo_final = entrenar_knn_final(
    split["X_train"], split["y_train"],
    split["X_val"], split["y_val"],
    n_neighbors=k_knn,
)

y_pred = modelo_final.predict(split["X_test"])
metricas_test = {
    "accuracy": float(accuracy_score(split["y_test"], y_pred)),
    "f1_macro": float(f1_score(split["y_test"], y_pred, average="macro")),
}
print(f"KNN final (k={k_knn}) sobre test:")
print(json.dumps(metricas_test, indent=2))
print(classification_report(split["y_test"], y_pred, digits=4))

RUTA_MODELO = RAIZ / "models" / "modelo.joblib"
joblib.dump(modelo_final, RUTA_MODELO)

matriz = confusion_matrix(split["y_test"], y_pred, labels=CLASES_GESTOS)
fig, ax = plt.subplots(figsize=(7, 6))
imagen = ax.imshow(matriz, cmap="Blues")
ax.set_xticks(range(len(CLASES_GESTOS)), CLASES_GESTOS, rotation=45, ha="right")
ax.set_yticks(range(len(CLASES_GESTOS)), CLASES_GESTOS)
ax.set_xlabel("Predicción")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de confusión · KNN (k={k_knn}) · test")
umbral = matriz.max() / 2
for i in range(len(CLASES_GESTOS)):
    for j in range(len(CLASES_GESTOS)):
        ax.text(
            j, i, matriz[i, j], ha="center", va="center",
            color="white" if matriz[i, j] > umbral else "black",
        )
fig.colorbar(imagen)
fig.tight_layout()

RUTA_MATRIZ = RAIZ / "models" / "matriz_confusion.png"
fig.savefig(RUTA_MATRIZ, dpi=150)
plt.close(fig)


KNN final (k=3) sobre test:
{
  "accuracy": 1.0,
  "f1_macro": 1.0
}
              precision    recall  f1-score   support

      neutro     1.0000    1.0000    1.0000        60
       palma     1.0000    1.0000    1.0000        60
         paz     1.0000    1.0000    1.0000        60
      pulgar     1.0000    1.0000    1.0000        60
        puno     1.0000    1.0000    1.0000        60
     senalar     1.0000    1.0000    1.0000        60

    accuracy                         1.0000       360
   macro avg     1.0000    1.0000    1.0000       360
weighted avg     1.0000    1.0000    1.0000       360



In [11]:
metadata = {
    "modelo": "knn",
    "clase_sklearn": type(modelo_final.named_steps["clasificador"]).__name__,
    "hiperparametros": {"n_neighbors": k_knn, "escalado": "StandardScaler"},
    "features": f"{len(COLUMNAS_FEATURES)} landmarks normalizados con espejado de mano izquierda",
    "clases": CLASES_GESTOS,
    "semilla": 42,
    "dataset": metricas["dataset"],
    "metricas_test": metricas_test,
    "fecha": datetime.now().isoformat(timespec="seconds"),
    "motivo_seleccion": "Modelo base evaluado en el conjunto de prueba.",
}
RUTA_METADATA = RAIZ / "models" / "metadata.json"
RUTA_METADATA.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

# Medicion de latencia por muestra en CPU (< 5 ms)
muestra = np.zeros((1, len(COLUMNAS_FEATURES)), dtype=np.float32)
modelo_final.predict(muestra)
inicio = time.perf_counter()
for _ in range(1000):
    modelo_final.predict(muestra)
latencia_ms = (time.perf_counter() - inicio) * 1000.0 / 1000.0

metricas["evaluacion_test_knn"] = metricas_test
metricas["latencia_prediccion_ms"] = float(latencia_ms)
RUTA_METRICAS.write_text(json.dumps(metricas, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Modelo guardado en {RUTA_MODELO}")
print(f"Matriz de confusión en {RUTA_MATRIZ}")
print(f"Metadatos en {RUTA_METADATA}")
print(f"Latencia media de una muestra: {latencia_ms:.3f} ms")
assert latencia_ms < 5.0, "La inferencia debe tardar menos de 5 ms"
print("MODELO_VALIDO")

Modelo guardado en /home/kaleb/Escuela/Minería/Proyecto 2do Parc/models/modelo.joblib
Matriz de confusión en /home/kaleb/Escuela/Minería/Proyecto 2do Parc/models/matriz_confusion.png
Metadatos en /home/kaleb/Escuela/Minería/Proyecto 2do Parc/models/metadata.json
Latencia media de una muestra: 1.552 ms
MODELO_VALIDO


## Conclusiones del dataset base

El dataset cumple con las 2,400 muestras balanceadas (400 por clase) y 0% de valores faltantes.  A continuacion se analiza su comportamiento en camara real con ambas manos.

## 7. Adaptación al dominio y modelo real de la demo (`RandomForest`)

Al llevar el KNN académico (`k=3`) a la cámara en vivo con otra mano y otro ángulo, el acierto cayó a **18.5%** por cambio de dominio. Para la demo se capturaron **689 cuadros** reales de ambas manos (`570` base + `119` de corrección izquierda) y se entrenó un `RandomForestClassifier` sobre **64 características geométricas** (`models/modelo_rf_vivo.joblib`).

In [12]:
from scripts.filtro import FiltroDisparo, PestilloGesto
from scripts.utils import normalizar

metricas_vivo = json.loads((RAIZ / 'models' / 'metricas_rf_vivo.json').read_text(encoding='utf-8'))
resumen_vivo = pd.DataFrame([
    {'Evaluación': 'Base 570 · mano izquierda (cross-hand)', 'Muestras': 315, 'Accuracy': metricas_vivo['validacion_entre_manos_base_570']['izquierda']['accuracy']},
    {'Evaluación': 'Base 570 · mano derecha (cross-hand)', 'Muestras': 255, 'Accuracy': metricas_vivo['validacion_entre_manos_base_570']['derecha']['accuracy']},
    {'Evaluación': 'Modelo final 689 · split temporal 70/30', 'Muestras': metricas_vivo['validacion_temporal_689']['muestras_test'], 'Accuracy': metricas_vivo['validacion_temporal_689']['accuracy']},
    {'Evaluación': 'Holdout corrección izquierda (30%)', 'Muestras': metricas_vivo['holdout_correccion_30']['muestras_test'], 'Accuracy': metricas_vivo['holdout_correccion_30']['accuracy']},
    {'Evaluación': 'Sesión independiente en vivo', 'Muestras': metricas_vivo['sesion_190']['muestras'], 'Accuracy': metricas_vivo['sesion_190']['accuracy']},
    {'Evaluación': 'Validación guiada en vivo (solo acciones)', 'Muestras': 455, 'Accuracy': metricas_vivo['validacion_en_vivo']['exactitud_solo_acciones']},
])
print(f"Latencia mediana RF en CPU: {metricas_vivo['latencia_ms']['mediana']:.2f} ms (máx {metricas_vivo['latencia_ms']['max']:.2f} ms)")
resumen_vivo

Latencia mediana RF en CPU: 7.75 ms (máx 10.02 ms)


,Evaluación,Muestras,Accuracy
0,Base 570 · mano izquierda (cross-hand),315,0.9302
1,Base 570 · mano derecha (cross-hand),255,0.9608
2,Modelo final 689 · split temporal 70/30,211,0.9858
3,Holdout corrección izquierda (30%),36,0.8056
4,Sesión independiente en vivo,190,1.0000
5,Validación guiada en vivo (solo acciones),455,0.9341


In [13]:
import warnings
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings('ignore', category=InconsistentVersionWarning)

# Replay de la sesión independiente (190 cuadros) con el flujo real de la demo:
# disparo en el primer cuadro + PestilloGesto anti-repetición + cooldown de 0.5 s.
modelo_vivo = joblib.load(RAIZ / 'models' / 'modelo_rf_vivo.joblib')
sesion_ind = pd.read_csv(RAIZ / 'data' / 'vivo' / 'sesion_independiente.csv')
filtro_demo = FiltroDisparo(frames_confirmacion=1, cooldown_s=0.5)
pestillo_demo = PestilloGesto()
acciones_disparadas = []
aciertos_ind = 0
reloj = 0.0

for _, fila in sesion_ind.iterrows():
    pts = np.array([
        fila[[f'x{i}' for i in range(21)]],
        fila[[f'y{i}' for i in range(21)]],
        fila[[f'z{i}' for i in range(21)]],
    ], dtype=np.float32).T
    vec = normalizar(pts, str(fila['handedness']))
    pred = str(modelo_vivo.predict(vec.reshape(1, -1))[0])
    aciertos_ind += int(pred == fila['label'])
    reloj += 1 / 15
    if pestillo_demo.actualizar(pred):
        disp = filtro_demo.actualizar(pred, reloj)
    else:
        filtro_demo.reiniciar()
        disp = None
    if disp is not None:
        pestillo_demo.bloquear(disp)
        acciones_disparadas.append(disp)

print(f'Exactitud en sesión independiente: {aciertos_ind}/{len(sesion_ind)} ({aciertos_ind / len(sesion_ind):.1%})')
print(f'Acciones disparadas por bloque (1 por gesto): {acciones_disparadas}')

Exactitud en sesión independiente: 190/190 (100.0%)
Acciones disparadas por bloque (1 por gesto): ['palma', 'senalar', 'paz', 'pulgar', 'puno']


## 8. Integracion en la demo en tiempo real (`demo.py`)

En `scripts/demo.py` conectamos la camara, MediaPipe, el modelo `RandomForest` y `playerctl` para controlar Spotify, usando `PestilloGesto` para ejecutar una sola accion por gesto.

In [14]:
def mostrar_bloque(ruta: Path, desde: int, hasta: int) -> None:
    lineas = ruta.read_text(encoding='utf-8').splitlines()
    for numero, linea in enumerate(lineas[desde - 1:hasta], start=desde):
        print(f'{numero:4d} | {linea}')

print('scripts/demo.py · predicción y pestillo')
mostrar_bloque(RAIZ / 'scripts' / 'demo.py', 248, 271)

scripts/demo.py · predicción y pestillo
 248 |                 vector = normalizar(puntos, handedness)
 249 |                 gesto_ml = str(modelo.predict(vector.reshape(1, -1))[0])
 250 |                 gesto_geo = clasificar_geometrico(puntos)
 251 |                 gesto = gesto_geo if args.usar_regla else gesto_ml
 252 |             else:
 253 |                 suavizador.reiniciar()
 254 |                 gesto_ml = None
 255 |                 gesto_geo = None
 256 |                 gesto = None
 257 | 
 258 |             if pestillo.actualizar(gesto):
 259 |                 disparo = filtro.actualizar(gesto, ahora)
 260 |             else:
 261 |                 filtro.reiniciar()
 262 |                 disparo = None
 263 | 
 264 |             if disparo is not None:
 265 |                 ultima_accion = f"{disparo} → {DESCRIPCION_ACCION[disparo]}"
 266 |                 if args.dry_run:
 267 |                     print(f"[DRY-RUN] {ultima_accion}", flush=True)
 268 |        

In [15]:
print('scripts/filtro.py · PestilloGesto')
mostrar_bloque(RAIZ / 'scripts' / 'filtro.py', 51, 82)

scripts/filtro.py · PestilloGesto
  51 |     def actualizar(self, gesto: Optional[str]) -> bool:
  52 |         """Actualiza el pestillo y devuelve True si permite un nuevo disparo."""
  53 |         if self._gesto is None:
  54 |             return True
  55 | 
  56 |         if gesto in (None, "neutro"):
  57 |             self._neutro += 1
  58 |             self._cambio = None
  59 |             self._contador_cambio = 0
  60 |             if self._neutro >= self.cuadros_neutro:
  61 |                 self._gesto = None
  62 |                 self._neutro = 0
  63 |             return False
  64 | 
  65 |         self._neutro = 0
  66 |         if gesto == self._gesto:
  67 |             self._cambio = None
  68 |             self._contador_cambio = 0
  69 |             return False
  70 | 
  71 |         if gesto == self._cambio:
  72 |             self._contador_cambio += 1
  73 |         else:
  74 |             self._cambio = gesto
  75 |             self._contador_cambio = 1
 

**Comandos para la demo en vivo:**

```bash
python scripts/demo.py --dry-run --timeout 5
python scripts/demo.py
pytest tests/ -q
```

### Limitaciones que hay que decir

- El modelo está calibrado para **un usuario y una cámara**; no se afirma que generalice.
- El `neutro` izquierdo se confunde con `palma` en 9/51 cuadros: el pestillo evita repeticiones, pero una mano fugaz puede accionar una vez.
- La validación independiente de 190 cuadros es la evidencia fuerte; el F1=1.0 del dataset académico es una cota optimista por cuadros consecutivos casi idénticos.